###L04: PySpark DataFrame essentials on the bronze data

####Step 1. Load and inspect

In [0]:
from pyspark.sql import functions as F
df = spark.table("da.bronze.yellow_trips")
df.printSchema()
print(df.count())
display(df.limit(10))

SQL Equivalent

In [0]:
%sql
DESCRIBE TABLE da.bronze.yellow_trips;
SELECT COUNT(*) FROM da.bronze.yellow_trips;
SELECT * FROM da.bronze.yellow_trips LIMIT 10;

###Step 2. Select, filter and derive
Build a small DataFrame of long trips. Nothing runs until display() or count() is called.

In [0]:
long_trips = (df.select("tpep_pickup_datetime", "tpep_dropoff_datetime", "trip_distance", "fare_amount", "PULocationID")
                .filter(F.col("trip_distance") > 10)
                .withColumn("fare_per_mile", F.round(F.col("fare_amount") / F.col("trip_distance"), 2)))
display(long_trips.limit(10))

SQL Equivalent

In [0]:
%sql
SELECT tpep_pickup_datetime, tpep_dropoff_datetime, trip_distance, fare_amount, PULocationID,
    ROUND(fare_amount/trip_distance,2) AS fare_per_mile
FROM da.bronze.yellow_trips
WHERE trip_distance > 10
LIMIT 10
;

###Step 3. Aggregate
Average fare and trip count by pickup zone, top 10 busiest zones.

In [0]:
top = (df.groupBy("PULocationID")
         .agg(F.count("*").alias("trips"), F.round(F.avg("fare_amount"), 2).alias("avg_fare"))
         .orderBy(F.desc("trips")).limit(10))
display(top)

SQL equivalent

In [0]:
%sql
SELECT PULocationID, COUNT(*) AS trips, round(avg(fare_amount),2) AS avg_fare
FROM da.bronze.yellow_trips
GROUP BY 1
ORDER BY COUNT(*) DESC
LIMIT 10
;


###Step 4. Profile nulls
Count nulls per column. This tells you which columns need cleaning rules. This is better in Python; it can be done in SQL, but you would need to check each column individually.

In [0]:
nulls = df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns])
display(nulls)

   
SQL Equivalent — one expression per column. Python loops over df.columns; SQL needs each column written out explicitly.

In [0]:
%sql
SELECT
  count_if(VendorID IS NULL)               AS VendorID,
  count_if(tpep_pickup_datetime IS NULL)    AS tpep_pickup_datetime,
  count_if(tpep_dropoff_datetime IS NULL)   AS tpep_dropoff_datetime,
  count_if(passenger_count IS NULL)         AS passenger_count,
  count_if(trip_distance IS NULL)           AS trip_distance,
  count_if(RatecodeID IS NULL)              AS RatecodeID,
  count_if(store_and_fwd_flag IS NULL)      AS store_and_fwd_flag,
  count_if(PULocationID IS NULL)            AS PULocationID,
  count_if(DOLocationID IS NULL)            AS DOLocationID,
  count_if(payment_type IS NULL)           AS payment_type,
  count_if(fare_amount IS NULL)            AS fare_amount,
  count_if(extra IS NULL)                   AS extra,
  count_if(mta_tax IS NULL)                 AS mta_tax,
  count_if(tip_amount IS NULL)              AS tip_amount,
  count_if(tolls_amount IS NULL)           AS tolls_amount,
  count_if(improvement_surcharge IS NULL)   AS improvement_surcharge,
  count_if(total_amount IS NULL)            AS total_amount,
  count_if(congestion_surcharge IS NULL)    AS congestion_surcharge,
  count_if(Airport_fee IS NULL)             AS Airport_fee,
  count_if(source_file IS NULL)            AS source_file,
  count_if(ingested_at IS NULL)             AS ingested_at
FROM da.bronze.yellow_trips;

###Step 5. Read the plan
explain() shows the physical plan; look for Exchange (a shuffle) caused by groupBy.

In [0]:
top.explain()

###Step 6. Transformation or action?
For each line in your notebook, write T (transformation) or A (action) in a markdown cell. select, filter, withColumn, groupBy, orderBy are T; count, display, write are A.

from pyspark.sql import functions as F          
df = spark.table("da.bronze.yellow_trips")  => T
df.printSchema()                            => A
print(df.count())                           => A    
display(df.limit(10))                       => A

long_trips = (df.select("tpep_pickup_datetime", "tpep_dropoff_datetime", "trip_distance", "fare_amount", "PULocationID")
                .filter(F.col("trip_distance") > 10)
                .withColumn("fare_per_mile", F.round(F.col("fare_amount") / F.col("trip_distance"), 2)))                    => T
display(long_trips.limit(10))                                                                                               => A

top = (df.groupBy("PULocationID")
         .agg(F.count("*").alias("trips"), F.round(F.avg("fare_amount"), 2).alias("avg_fare"))
         .orderBy(F.desc("trips")).limit(10))                                                       => T
display(top)                                                                                        => A

nulls = df.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df.columns])             => T
display(nulls)                                                                                      => A

top.explain()           => A